In [ ]:
!cp 

In [1]:
"""
LATUP-Net Training Script - Paper Setup
==============================================

This script replicates the exact setup from the LATUP-Net paper:
- Input: 128x128x128x4 (4 modalities: FLAIR, T1, T1CE, T2)
- Output: 128x128x128x4 (4 classes: BG, NCR/NET, ED, ET)
- Batch size: 1
- LeakyReLU alpha: 0.1
- Optimizer: Adam (beta1=0.9, beta2=0.999)
- Learning rate: 1e-4
- Epochs: 200
- Loss: Weighted Dice Score Loss (WDL) from repo
- Dropout: 0.2
- L2 Regularization: 0.02
- Output activation: Softmax

BraTS Metrics:
- WT (Whole Tumor): Classes 1+2+3 (all tumor)
- TC (Tumor Core): Classes 1+3 (NCR/NET + ET)
- ET (Enhancing Tumor): Class 3 only
"""

'\nLATUP-Net Training Script - Paper Setup\n==============================================\n\nThis script replicates the exact setup from the LATUP-Net paper:\n- Input: 128x128x128x4 (4 modalities: FLAIR, T1, T1CE, T2)\n- Output: 128x128x128x4 (4 classes: BG, NCR/NET, ED, ET)\n- Batch size: 1\n- LeakyReLU alpha: 0.1\n- Optimizer: Adam (beta1=0.9, beta2=0.999)\n- Learning rate: 1e-4\n- Epochs: 200\n- Loss: Weighted Dice Score Loss (WDL) from repo\n- Dropout: 0.2\n- L2 Regularization: 0.02\n- Output activation: Softmax\n\nBraTS Metrics:\n- WT (Whole Tumor): Classes 1+2+3 (all tumor)\n- TC (Tumor Core): Classes 1+3 (NCR/NET + ET)\n- ET (Enhancing Tumor): Class 3 only\n'

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger

2025-12-05 01:53:27.936802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764899608.272051      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764899608.367961      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

### Configuration

In [3]:
DATA_PATH = "/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
PREPROCESSED_DIR = "/kaggle/working/pre_brats/"
CHECKPOINT_DIR = "/kaggle/working/checkpoints/"

modalities = ["flair", "t1", "t1ce", "t2"]
num_classes = 4
target_shape = (128, 128, 128)
batch_size = 1  
total_epochs = 50

# Paper hyperparameters
lrelu_alpha = 0.1 
learning_rate = 1e-4 
dropout_rate = 0.2 
l2_reg = 0.02 
adam_beta1 = 0.9
adam_beta2 = 0.999

# Enable memory growth for GPUs
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

2025-12-05 01:53:50.392938: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### Data Loading Functions

In [ ]:
def normalize(img):
    """Normalize MRI image."""
    img = img.astype(np.float32)
    img = (img - np.mean(img)) / (np.std(img) + 1e-8)
    return img


def resize_volume(volume, target_shape):
    """Resize 3D volume using linear interpolation."""
    factors = [t / s for t, s in zip(target_shape, volume.shape[:3])]
    return zoom(volume, factors, order=1)


def resize_mask(mask, target_shape):
    """Resize mask using nearest neighbor interpolation."""
    if mask.ndim == 4:
        factors = [t / s for t, s in zip(target_shape, mask.shape[:3])] + [1]
    elif mask.ndim == 3:
        factors = [t / s for t, s in zip(target_shape, mask.shape)]
    else:
        raise RuntimeError(f"Unexpected mask shape: {mask.shape}")
    mask_resized = zoom(mask, factors, order=0)
    return mask_resized.astype(np.uint8)


def map_brats_labels(mask):
    """
    Map BraTS labels to 0..3.
    Original: 0=BG, 1=NCR/NET, 2=ED, 4=ET
    Mapped: 0=BG, 1=NCR/NET, 2=ED, 3=ET
    """
    mapped = np.zeros_like(mask, dtype=np.int32)
    mapped[mask == 1] = 1  
    mapped[mask == 2] = 2 
    mapped[mask == 4] = 3 
    return mapped


def load_patient(patient_id, data_path=DATA_PATH):
    """Load and preprocess patient images (4 modalities)."""
    images = []
    patient_folder = os.path.join(data_path, patient_id)
    for mod in modalities:
        path = os.path.join(patient_folder, f"{patient_id}_{mod}.nii")
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing {path}")
        img = nib.load(path).get_fdata()
        img = resize_volume(img, target_shape)
        img = normalize(img)
        images.append(img)
    return np.stack(images, axis=-1).astype(np.float32)


def load_mask(patient_id, data_path=DATA_PATH):
    """Load and preprocess patient mask."""
    mask_path = os.path.join(data_path, patient_id, f"{patient_id}_seg.nii")
    if not os.path.exists(mask_path):
        raise FileNotFoundError(f"Missing mask: {mask_path}")
    mask = nib.load(mask_path).get_fdata().astype(np.int32)
    mask = map_brats_labels(mask)
    mask = resize_mask(mask, target_shape)
    mask_onehot = tf.one_hot(mask, depth=num_classes)
    return tf.cast(mask_onehot, tf.float32)  


def preprocess_and_save_npz(patient_id):
    """Preprocess and save patient data as npz."""
    x = load_patient(patient_id)
    y = load_mask(patient_id).numpy()
    np.savez_compressed(os.path.join(PREPROCESSED_DIR, f"{patient_id}.npz"), x=x, y=y)


### Calculate Class Weights for WDL

In [ ]:
def calculate_class_weights_for_wdl(patient_ids, sample_size=100):
    """
    Calculate class weights for Weighted Dice Score Loss.
    Uses inverse frequency weighting.
    """
    class_counts = np.zeros(num_classes, dtype=np.float32)
    sample_ids = np.random.choice(
        patient_ids, min(sample_size, len(patient_ids)), replace=False
    )

    for pid in sample_ids:
        try:
            d = np.load(os.path.join(PREPROCESSED_DIR, f"{pid}.npz"))
            y = d["y"]
            y_classes = np.argmax(y, axis=-1)
            unique, counts = np.unique(y_classes, return_counts=True)
            for u, c in zip(unique, counts):
                class_counts[u] += c
        except:
            continue

    total = np.sum(class_counts)
    if total > 0:
        # Inverse frequency weighting
        weights = total / (num_classes * class_counts + 1e-6)
        # Normalize
        weights = weights / np.sum(weights) * num_classes
    else:
        # Fallback weights (approximate from BraTS data)
        weights = np.array([0.2, 1.0, 1.2, 1.5], dtype=np.float32)

    return weights.astype(np.float32)


### BraTS Metrics: WT, TC, ET

In [ ]:
def calculate_wt_tc_et_dsc(y_true, y_pred):
    """
    Calculate BraTS metrics: WT, TC, ET DSC.

    WT (Whole Tumor) = Classes 1+2+3 (all tumor)
    TC (Tumor Core) = Classes 1+3 (NCR/NET + ET)
    ET (Enhancing Tumor) = Class 3 only

    Args:
        y_true: True mask (H, W, D, 4) or class indices (H, W, D)
        y_pred: Predicted mask (H, W, D, 4) or class indices (H, W, D)

    Returns:
        Dictionary with 'WT', 'TC', 'ET' DSC scores
    """
    smooth = 1e-6

    # Convert to class indices if one-hot
    if y_true.ndim == 4:
        y_true = np.argmax(y_true, axis=-1)
    if y_pred.ndim == 4:
        y_pred = np.argmax(y_pred, axis=-1)

    # WT: Whole Tumor = classes 1, 2, 3
    wt_true = ((y_true == 1) | (y_true == 2) | (y_true == 3)).astype(np.float32)
    wt_pred = ((y_pred == 1) | (y_pred == 2) | (y_pred == 3)).astype(np.float32)
    wt_intersection = np.sum(wt_true * wt_pred)
    wt_union = np.sum(wt_true) + np.sum(wt_pred)
    wt_dsc = (2.0 * wt_intersection + smooth) / (wt_union + smooth)

    # TC: Tumor Core = classes 1, 3 
    tc_true = ((y_true == 1) | (y_true == 3)).astype(np.float32)
    tc_pred = ((y_pred == 1) | (y_pred == 3)).astype(np.float32)
    tc_intersection = np.sum(tc_true * tc_pred)
    tc_union = np.sum(tc_true) + np.sum(tc_pred)
    tc_dsc = (2.0 * tc_intersection + smooth) / (tc_union + smooth)

    # ET: Enhancing Tumor = class 3 only
    et_true = (y_true == 3).astype(np.float32)
    et_pred = (y_pred == 3).astype(np.float32)
    et_intersection = np.sum(et_true * et_pred)
    et_union = np.sum(et_true) + np.sum(et_pred)
    et_dsc = (2.0 * et_intersection + smooth) / (et_union + smooth)

    return {"WT": float(wt_dsc), "TC": float(tc_dsc), "ET": float(et_dsc)}


class BraTSMetrics(tf.keras.callbacks.Callback):
    """Callback to calculate WT, TC, ET metrics during validation."""

    def __init__(self, val_dataset, val_steps):
        super().__init__()
        self.val_dataset = val_dataset
        self.val_steps = val_steps
        self.wt_scores = []
        self.tc_scores = []
        self.et_scores = []

    def on_epoch_end(self, epoch, logs=None):
        """Calculate WT, TC, ET metrics at end of epoch."""
        wt_dscs = []
        tc_dscs = []
        et_dscs = []

        for batch_idx, (x_batch, y_batch) in enumerate(self.val_dataset):
            if batch_idx >= self.val_steps:
                break

            # Get predictions
            y_pred = self.model.predict(x_batch, verbose=0)

            # Calculate metrics for this batch
            for i in range(y_batch.shape[0]):
                metrics = calculate_wt_tc_et_dsc(y_batch[i], y_pred[i])
                wt_dscs.append(metrics["WT"])
                tc_dscs.append(metrics["TC"])
                et_dscs.append(metrics["ET"])

        # Average over validation set
        avg_wt = np.mean(wt_dscs) if wt_dscs else 0.0
        avg_tc = np.mean(tc_dscs) if tc_dscs else 0.0
        avg_et = np.mean(et_dscs) if et_dscs else 0.0

        self.wt_scores.append(avg_wt)
        self.tc_scores.append(avg_tc)
        self.et_scores.append(avg_et)

        logs = logs or {}
        logs["val_WT_DSC"] = avg_wt
        logs["val_TC_DSC"] = avg_tc
        logs["val_ET_DSC"] = avg_et

        print(f"\nBraTS Metrics - WT: {avg_wt:.4f}, TC: {avg_tc:.4f}, ET: {avg_et:.4f}")


### Main Training Script

In [ ]:

def main():
    # Load data
    csv_path = os.path.join(DATA_PATH, "name_mapping.csv")
    df = pd.read_csv(csv_path)
    patients = df["BraTS_2020_subject_ID"].tolist()
    train_patients, val_patients = train_test_split(
        patients, test_size=0.2, random_state=42
    )

    # Create directories
    os.makedirs(PREPROCESSED_DIR, exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # Preprocess data
    print("Preprocessing data...")
    for pid in patients:
        try:
            if not os.path.exists(os.path.join(PREPROCESSED_DIR, f"{pid}.npz")):
                preprocess_and_save_npz(pid)
        except FileNotFoundError:
            continue
    print("Preprocessing complete!")

    # Filter patients
    def load_mask_path(patient_id):
        return f"{DATA_PATH}/{patient_id}/{patient_id}_seg.nii"

    train_patients_filtered = [
        p for p in train_patients if os.path.exists(load_mask_path(p))
    ]
    val_patients_filtered = [
        p for p in val_patients if os.path.exists(load_mask_path(p))
    ]

    # Split train/val
    num_train = int(0.8 * len(train_patients_filtered))
    train_ids = train_patients_filtered[:num_train]
    val_ids = train_patients_filtered[num_train:]

    print(f"Train IDs: {len(train_ids)}")
    print(f"Val IDs: {len(val_ids)}")

    # Calculate class weights for Weighted Dice Score Loss
    class_weights_array = calculate_class_weights_for_wdl(train_ids, sample_size=100)
    print(f"Calculated class weights: {class_weights_array}")

    # Create Weighted Dice Score Loss (WDL) from repo
    sys.path.append("/kaggle/working/")
    from latup_net.loss import WeightedDiceScore

    # WDL format: weight0 - sum(weight_i * DSC_i)
    wdl_weights = [
        (class_weights_array[0], 0),  
        (class_weights_array[1], 1), 
        (class_weights_array[2], 2),
        (class_weights_array[3], 3), 
    ]
    weight0 = 1.0
    loss_fn = WeightedDiceScore(weight0=weight0, weights=wdl_weights, epsilon=0.00001)

    # Create datasets
    def fast_npz_generator(patient_list):
        for pid in patient_list:
            d = np.load(os.path.join(PREPROCESSED_DIR, f"{pid}.npz"))
            yield d["x"], d["y"]

    train_dataset = tf.data.Dataset.from_generator(
        lambda: fast_npz_generator(train_ids),
        output_signature=(
            tf.TensorSpec(shape=(128, 128, 128, 4), dtype=tf.float32),
            tf.TensorSpec(shape=(128, 128, 128, 4), dtype=tf.float32),
        ),
    )

    train_dataset = train_dataset.shuffle(
        buffer_size=min(32, len(train_ids)), reshuffle_each_iteration=True
    )
    train_dataset = train_dataset.batch(batch_size)
    train_dataset = train_dataset.repeat()
    train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_generator(
        lambda: fast_npz_generator(val_ids),
        output_signature=(
            tf.TensorSpec(shape=(128, 128, 128, 4), dtype=tf.float32),
            tf.TensorSpec(shape=(128, 128, 128, 4), dtype=tf.float32),
        ),
    )
    val_dataset = val_dataset.batch(batch_size).prefetch(4)

    # Setup callbacks
    steps_per_epoch = len(train_ids) // batch_size
    val_steps = len(val_ids) // batch_size

    model_checkpoint = ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, "best_model.h5"),
        monitor="val_dsc", 
        save_best_only=True,
        save_weights_only=False,
        mode="max",
        verbose=1,
    )

    csv_logger = CSVLogger(os.path.join(CHECKPOINT_DIR, "training_log.csv"))

    # BraTS metrics callback
    brats_metrics = BraTSMetrics(val_dataset, val_steps)

    callbacks = [model_checkpoint, csv_logger, brats_metrics]

    # Create optimizer
    optimizer_fn = lambda batch_size: Adam(
        learning_rate=learning_rate, beta_1=adam_beta1, beta_2=adam_beta2
    )

    # Import LATUP-Net
    from latup_net.latupnet import LATUPNet
    from latup_net.trainer import dsc, iou

    def get_metrics():
        return [dsc, iou]

    # Create model (exact paper settings)
    latup_model_class = LATUPNet(
        name="LATUPNet_BraTS_Paper",
        loss=loss_fn,  
        attention="SE", 
        lrelu_alpha=lrelu_alpha, 
        metrics=get_metrics,
        dropout=dropout_rate,  
        optimiser=optimizer_fn,
        l2_reg=l2_reg, 
    )

    model = latup_model_class.construct(
        seq=(128, 128, 128, 4, num_classes), batch_size=batch_size
    )

    model.summary()

    # Train
    print("\n" + "=" * 60)
    print("LATUP-NET TRAINING - EXACT PAPER SETUP")
    print("=" * 60)
    print(f"Input shape: 128x128x128x4 (4 modalities)")
    print(f"Output shape: 128x128x128x4 (4 classes)")
    print(f"Batch size: {batch_size}")
    print(f"Learning rate: {learning_rate}")
    print(f"Epochs: {total_epochs}")
    print(f"Loss: Weighted Dice Score Loss (WDL)")
    print(f"Dropout: {dropout_rate}")
    print(f"L2 Regularization: {l2_reg}")
    print(f"LeakyReLU alpha: {lrelu_alpha}")
    print(f"Steps per epoch: {steps_per_epoch}")
    print(f"Validation steps: {val_steps}")
    print(f"Class weights: {class_weights_array}")
    print("=" * 60 + "\n")

    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

    history = model.fit(
        train_dataset,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_dataset,
        validation_steps=val_steps,
        epochs=total_epochs,
        callbacks=callbacks,
        verbose=1,
    )

    # Print best results
    best_epoch = np.argmax(history.history["val_dsc"])
    print("\n" + "=" * 60)
    print("TRAINING COMPLETE")
    print("=" * 60)
    print(f"Best Epoch: {best_epoch + 1}")
    print(f"Best Val DSC: {history.history['val_dsc'][best_epoch]:.4f}")
    print(f"Best Val IoU: {history.history['val_iou'][best_epoch]:.4f}")
    print(f"Best Val Loss: {history.history['val_loss'][best_epoch]:.4f}")

    # Print BraTS metrics
    if brats_metrics.wt_scores:
        best_wt_epoch = np.argmax(brats_metrics.wt_scores)
        best_tc_epoch = np.argmax(brats_metrics.tc_scores)
        best_et_epoch = np.argmax(brats_metrics.et_scores)
        print(f"\nBraTS Metrics (Best):")
        print(
            f"  WT DSC: {brats_metrics.wt_scores[best_wt_epoch]:.4f} (epoch {best_wt_epoch+1})"
        )
        print(
            f"  TC DSC: {brats_metrics.tc_scores[best_tc_epoch]:.4f} (epoch {best_tc_epoch+1})"
        )
        print(
            f"  ET DSC: {brats_metrics.et_scores[best_et_epoch]:.4f} (epoch {best_et_epoch+1})"
        )
        print(f"\nBraTS Metrics (Final):")
        print(f"  WT DSC: {brats_metrics.wt_scores[-1]:.4f}")
        print(f"  TC DSC: {brats_metrics.tc_scores[-1]:.4f}")
        print(f"  ET DSC: {brats_metrics.et_scores[-1]:.4f}")

    print("=" * 60)

    return model, history


In [ ]:
model, history = main()